# 05. Production Pipeline

End-to-end inference demo: load a pretrained model, process a new audio file, output anomaly score.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from models.ensemble import HybridAnomalyDetector
from models.classical_features import extract_classical_features
from evaluation.metrics import evaluate_detector

## 1. Train and Save a Model

In [ ]:
# Train a simple GMM on synthetic data
np.random.seed(42)
X_train = np.random.randn(100, 100)  # 100 normal samples

detector = HybridAnomalyDetector(method='gmm', n_components=8)
detector.fit(X_train)
detector.save('/tmp/demo_model.pkl')
print('Model saved to /tmp/demo_model.pkl')

## 2. Load Model and Score New Sample

In [ ]:
# Load model and score a new sample
loaded = HybridAnomalyDetector.load('/tmp/demo_model.pkl')
X_new = np.random.randn(1, 100)
score = float(loaded.score_samples(X_new)[0])

import math
normalized = 1.0 / (1.0 + math.exp(-score))
label = 'anomaly' if normalized > 0.5 else 'normal'
print(f'Raw score: {score:.4f}')
print(f'Normalized score: {normalized:.4f}')
print(f'Classification: {label}')

## 3. Threshold Tuning

In [ ]:
# Generate test data with labels
np.random.seed(1)
X_test = np.random.randn(60, 100)
y_test = np.array([0]*40 + [1]*20)  # 40 normal, 20 anomaly

scores = loaded.score_samples(X_test)
result = evaluate_detector(y_test, scores)
print(f'AUC: {result.auc:.4f}')
print(f'Average Precision: {result.average_precision:.4f}')
print(f'Optimal threshold: {result.threshold:.4f}')

## 4. Production Usage

```bash
# Score a new audio file
python scripts/inference.py \
    --model models/pump_hybrid_gmm16.pkl \
    --audio test_sample.wav
# → Anomaly score: 0.823 (likely anomaly)
```